# Extract data from CERCA raw files

In [4]:
from docx import Document
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [5]:
interest_centers = ['BETA', 'CREAF', 'ICN2', 'ISGlobal', 'ResearchMar']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'

## Extract DOI list

**Each file has a different format so we go center by center**

### BETA

In [6]:
center_name = 'BETA/'
file_name = 'LIST OF SCIENTIFIC PUBLICATIONS_BETA'

In [7]:
doc = Document(file_path + 'BM_' + center_name + file_name + '.docx')
pubs = [para.text for para in doc.paragraphs]
DOI_tmp = [pub.split('DOI:', 1)[1].strip() for pub in pubs  if 'DOI:' in pub]
DOI_BETA = [(DOI.split('https://doi.org/', 1)[1].strip() if 'https://doi.org/' in DOI else DOI) for DOI in DOI_tmp]
df_BETA = pd.DataFrame(DOI_BETA, columns = ['DOI'])
df_BETA['Center'] = 'BETA'
df_BETA

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
63,10.1016/j.bcab.2021.102114,BETA
64,10.1016/j.sajb.2021.06.035,BETA
65,10.1002/ecy.3614,BETA
66,10.3390/ijms222011277,BETA


### CREAF

In [13]:
center_name = 'CREAF/'
file_name = 'Publicacions CREAF 2021-2024'

In [14]:
df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', skiprows = 5)
df_CREAF = df[['doi']].rename(columns = {'doi': 'DOI'})
df_CREAF['Center'] = 'CREAF'
df_CREAF

,DOI,Center
0,10.1038/s43705-021-00073-5,CREAF
1,10.7325/galemys.2022.n5,CREAF
2,10.1109/IGARSS53475.2024.10640524,CREAF
3,10.1109/IGARSS53475.2024.10641964,CREAF
4,10.1109/MetroAgriFor63043.2024.10948835,CREAF
...,...,...
928,10.18601/01245996.v24n47.12,CREAF
929,10.1038/s43247-021-00229-0,CREAF
930,10.34133/2022/9764982,CREAF
931,10.34133/remotesensing.0085,CREAF


### ICN2

In [15]:
center_name = 'ICN2/'
file_name = 'ICN2_DOIs_Pubs2021-2024_CERCA2025'

In [16]:
df_ICN2 = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx').rename(columns = {'Doi': 'DOI'})
df_ICN2['Center'] = 'ICN2'
df_ICN2

,DOI,Center
0,10.59277/ROMJIST.2024.2.09,ICN2
1,10.1002/ece2.12,ICN2
2,10.1093/mam/ozae044.520,ICN2
3,10.1103/physrevlett.132.266301,ICN2
4,10.7203/metode.15.27225,ICN2
...,...,...
823,10.1103/PhysRevB.108.054524,ICN2
824,10.1021/acsaem.1c02919,ICN2
825,10.1016/j.synthmet.2021.116844,ICN2
826,10.1038/s42254-021-00318-1,ICN2


### ISGlobal

**NOT PROPERLY TREATED, THE XLSX HAS MORE THAN ONE SHEET**

In [22]:
center_name = 'ISGlobal/'
file_name = 'ISGlobal_DOIs Publications_2021-2024'

In [23]:
df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', skiprows = 1)
df_ISGlobal = df[['Doi']].rename(columns = {'Doi':'DOI'})
df_ISGlobal['Center'] = 'ISGlobal'
df_ISGlobal

,DOI,Center
0,10.1038/s41380-019-0558-2,ISGlobal
1,10.1016/j.edumed.2019.09.004,ISGlobal
2,10.1038/ng.2238,ISGlobal
3,10.1111/all.14422,ISGlobal
4,10.1016/j.nrl.2020.02.006,ISGlobal
...,...,...
753,10.3390/ijms222413363,ISGlobal
754,10.3390/pathogens10121588,ISGlobal
755,10.1371/journal.pntd.0009954,ISGlobal
756,10.1136/bmjopen-2021-052897,ISGlobal


### ResearchMar

In [18]:
center_name = 'ResearchMar/'
file_name = 'HMRIB_CERCA_DOIs21-24_VF'

In [19]:
df_ResearchMar = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', header = None)[0:-1].rename(columns = {0: 'DOI'})
df_ResearchMar['Center'] = 'ResearchMar'
df_ResearchMar

,DOI,Center
0,10.1001/jama.2021.15255,ResearchMar
1,10.1001/jama.2022.1645,ResearchMar
2,10.1001/jamacardio.2021.4690,ResearchMar
3,10.1001/jamacardio.2022.1988,ResearchMar
4,10.1001/jamadermatol.2022.0434,ResearchMar
...,...,...
5346,10.7759/cureus.13183,ResearchMar
5347,10.7759/cureus.16472,ResearchMar
5348,10.7759/cureus.40708,ResearchMar
5349,10.7759/cureus.62509,ResearchMar


In [20]:
df_centers = pd.concat([df_BETA, df_CREAF, df_ICN2, df_ISGlobal, df_ResearchMar], ignore_index = True)
df_centers.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS.csv', index = False)
df_centers

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
10081,10.7759/cureus.13183,ResearchMar
10082,10.7759/cureus.16472,ResearchMar
10083,10.7759/cureus.40708,ResearchMar
10084,10.7759/cureus.62509,ResearchMar


In [21]:
df_centers.DOI.nunique()

8820

## Check which publications are not in OA using DOI

In [27]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [28]:
in_query = str(tuple(df_centers.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wa.author_position,
       wa.is_corresponding,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """

df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,4210154894,ES
1,10.1038/s41591-023-02610-2,Charles Agyemang,47,middle,False,887064364,NL
2,10.1038/s41591-023-02610-2,Andrew Wong,750,middle,False,45129253,GB
3,10.1038/s41591-023-02610-2,Farhad Zamani,766,middle,False,161106909,IR
4,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,63634437,ES
...,...,...,...,...,...,...,...
189137,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210133907,FR
189138,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210097159,FR
189139,10.1111/jvh.13412,Karine Lacombe,33,middle,False,39804081,FR
189140,10.1002/lol2.10277,Maria Lundgren,33,middle,False,223464139,SE


In [29]:
(df_OA.DOI.nunique()) / df_centers.DOI.nunique()

0.9305475504322767

**Lets try to see if we can get them from Open aire**

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError

def extract_field(field):
    """Extract the first value of a list or $ value from a dict."""
    if isinstance(field, list):
        return field[0].get("$", None) if field else None
    elif isinstance(field, dict):
        return field.get("$", None)
    return field

BATCH_SIZE = 25
SLEEP_BETWEEN_REQUESTS = 1.0
BASE_URL = "https://api.openaire.eu/search/publications"
TIMEOUT_PER_DOI = 30           # max seconds per DOI request
TIMEOUT_PER_RESULT = 30        # max seconds per result processing

records_list = []
not_found_dois = []

df_missing = df_centers[~df_centers.DOI.isin(df_OA.DOI.unique())]

def process_single_result(result):
    """Process a single OpenAIRE result, extracting author-affiliation rows."""
    metadata = result.get("metadata", {}).get("oaf:entity", {}).get("oaf:result", {})
    doi_found = extract_field(metadata.get("pid"))
    authors = metadata.get("creator", [])
    rels = metadata.get("rels", {}).get("rel", [])
    doi_records = []

    for idx, author in enumerate(authors, start=1):
        author_firstname = extract_field(author.get("@name"))
        author_surname = extract_field(author.get("@surname"))
        author_name = f"{author_firstname} {author_surname}" if author_firstname or author_surname else None
        author_order = extract_field(author.get("@rank")) or idx

        author_rels = [rel for rel in rels if rel.get("to", {}).get("@class") == "hasAuthorInstitution"]
        if author_rels:
            for rel in author_rels:
                institution_id = rel["to"].get("$")
                country = rel.get("country", {}).get("@classname")
                doi_records.append({
                    "DOI": doi_found,
                    "author_name": author_name,
                    "author_order": author_order,
                    "institution_id": institution_id,
                    "COUNTRY_CODE": country
                })
        else:
            doi_records.append({
                "DOI": doi_found,
                "author_name": author_name,
                "author_order": author_order,
                "institution_id": None,
                "COUNTRY_CODE": None
            })

    return doi_records

# Process DOIs batch by batch
for i in tqdm(range(0, len(df_missing.DOI), BATCH_SIZE)):
    batch = df_missing.DOI[i:i+BATCH_SIZE]
    
    for doi in batch:
        try:
            # Fetch DOI with timeout
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(
                    requests.get,
                    BASE_URL,
                    params={"doi": doi, "format": "json", "size": 1},
                    timeout=30
                )
                try:
                    response = future.result(timeout=TIMEOUT_PER_DOI)
                    response.raise_for_status()
                    openaire_results = response.json()
                except TimeoutError:
                    print(f"⏱ Timeout fetching DOI {doi}, skipping.")
                    not_found_dois.append(doi)
                    continue

            results = openaire_results.get("response", {}).get("results", {}).get("result", [])
            if not results:
                not_found_dois.append(doi)
                continue
            if not isinstance(results, list):
                results = [results]

            # Process each result with its own timeout
            for result in results:
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(process_single_result, result)
                    try:
                        doi_records = future.result(timeout=TIMEOUT_PER_RESULT)
                        if doi_records:
                            records_list.extend(doi_records)
                    except TimeoutError:
                        print(f"⏱ Timeout processing a result for DOI {doi}, skipping this result.")

            time.sleep(SLEEP_BETWEEN_REQUESTS)

        except Exception as e:
            print(f"⚠️ Error processing DOI {doi}: {e}")
            not_found_dois.append(doi)
            

# Convert list to DataFrame once at the end
records = pd.DataFrame(records_list)
print("Number of DOIs not found:", len(set(not_found_dois)))
records = records.drop_duplicates(['DOI', 'institution_id', 'author_name']).reset_index(drop = True)
records


  0%|          | 0/23 [00:00<?, ?it/s]

⚠️ Error processing DOI 10.3390/su16010238.: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.5683/SP3/BIDMCI: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.1109/MetroAgriFor63043.2024.10948835: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.1017/S0007485324000567: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.70186/baeeZTOI5976: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.21138/GF.830: 'str' object has no attribute 'get'


 13%|█▎        | 3/23 [01:48<12:23, 37.16s/it]

⚠️ Error processing DOI 10.1098/rstb.2019.0810rstb20190810: 'NoneType' object has no attribute 'get'
⚠️ Error processing DOI 10.1016/S2352-3018(21)00051-5 : 'NoneType' object has no attribute 'get'


 17%|█▋        | 4/23 [02:22<11:25, 36.06s/it]

...

 87%|████████▋ | 20/23 [12:14<01:50, 36.72s/it]

⚠️ Error processing DOI 10.23736/S1973-9087.24.08234-0: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.25259/IJDVL_561_2022: 'str' object has no attribute 'get'
⚠️ Error processing DOI 10.25259/IJDVL_561_2022: 'str' object has no attribute 'get'


100%|██████████| 23/23 [13:51<00:00, 36.13s/it]


Number of DOIs not found: 25


,DOI,author_name,author_order,institution_id,COUNTRY_CODE
0,10.7818/ecos.2684,Meritxell Abril,1,openorgs____::6c3442fa797102c1c5c1a4319fce4119,Spain
1,10.7818/ecos.2684,Meritxell Abril,1,openorgs____::da01441f0598813c643e481cd22673d5,Spain
2,10.7818/ecos.2684,Isabel Muñoz,2,openorgs____::6c3442fa797102c1c5c1a4319fce4119,Spain
3,10.7818/ecos.2684,Isabel Muñoz,2,openorgs____::da01441f0598813c643e481cd22673d5,Spain
4,10.7818/ecos.2684,Margarita Menéndez,3,openorgs____::6c3442fa797102c1c5c1a4319fce4119,Spain
...,...,...,...,...,...
782759,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e1bc2739ac515ca512dc9e814cad9613,Italy
782760,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::38dea46ca400398cccbaf76ca0ba85c2,United States
782761,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e08612ddafc9e1533bb1c03b7a550a87,United States
782762,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::d41cf6bd4ab1b1362a44397e0b95c975,Italy


In [ ]:
df_DOI = pd.concat([df_OA, records.rename(columns = {'author_name' : 'display_name'})], ignore_index = True)
df_DOI.to_csv('../data/interim/CERCA_5_Bibliometria_SIRIS_processed.csv', index = False)
df_DOI

,DOI,display_name,author_order,institution_id,COUNTRY_CODE
0,10.1093/bjs/znab247,David Merlini,153,<NA>,None
1,10.1093/bjs/znab247,U. Novo Rivas,642,<NA>,None
2,10.1093/bjs/znab247,Basim Al-Khafaji,103,<NA>,None
3,10.1093/bjs/znab247,Vincenzo Papagni,663,<NA>,None
4,10.1038/s41467-024-49494-5,Edurne Martínez del Castillo,39,197323543,DE
...,...,...,...,...,...
634027,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e1bc2739ac515ca512dc9e814cad9613,Italy
634028,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::38dea46ca400398cccbaf76ca0ba85c2,United States
634029,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::e08612ddafc9e1533bb1c03b7a550a87,United States
634030,10.7554/elife.85893,Bernardo Rudy,11,openorgs____::d41cf6bd4ab1b1362a44397e0b95c975,Italy


In [ ]:
df_DOI[df_DOI.COUNTRY_CODE.isin(['ES', 'Spain'])].drop_duplicates(['display_name', 'institution_id']).dropna()

,DOI,display_name,author_order,institution_id,COUNTRY_CODE
5,10.1016/j.euroneuro.2023.06.003,Elena Rubio‐Abadal,37,4210127203,ES
7,10.1016/j.euroneuro.2023.06.003,Elena Rubio‐Abadal,37,4210128981,ES
10,10.1016/j.clgc.2022.06.001,Alejo Rodríguez‐Vida,35,4210130874,ES
27,10.3390/jcm10194402,J L Rueda García,44,2800562746,ES
32,10.1111/apt.18133,Raquel Mena,38,4210098336,ES
...,...,...,...,...,...
633962,10.7554/elife.81067,Study Oasis,31,pending_org_::97fcd65a2bc9fd4448e94200efe6a7ba,Spain
633964,10.7554/elife.81067,Study Oasis,31,openorgs____::08b006c8328e8d0320d28e35ed34b125,Spain
633967,10.7554/elife.81067,Study Oasis,31,cf__________::1170febd100770dd345d6ed983bf6c48,Spain
633968,10.7554/elife.81067,Study Oasis,31,pending_org_::04c48928af1d147d20064809e815b171,Spain


### Identify CERCA authors using OA (93% of the dataset)

- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation [2k affiliations]
  - String search with keywords for = 1 doi per raw affilation [4k affilations]

**We identify 70% of the provided DOI's**

For the ResearchMar the retrieval is 61%; being the center with most publications we will force the manual identification to increase it by selecting in a second phase using a manual validation [word mar and Barcelona]. We reach 67%

The problem is the hypothesis of using the parent affiliations (problem for the hospital) that is not good enough but I can't manually review all the affiliations of spain. So I can't improve it further

So to solve it we select the not found dois in Researchmar using the previous parent affiliations hypothesis and chec the raw affiliations in Spain looking for strings (mar, imim, parc combined with barcelona without checking).We reach 84%

In [30]:
cerca_centers = {'BETA' : [''], # NOT IN OA
                    'CREAF' : ['4210129656', '4401200259'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'BETA' : ['115304662'],
                    'CREAF' : ['123044942', '71999127'],
                    'ICN2' : ['123044942'],
                    'ISGlobal' : ['123044942', '170486558'],
                    'ResearchMar' : ['170486558']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(df_OA.institution_id.isin(parents_cerca)) & (~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
162,10.1016/j.ijid.2023.10.019,José Marı́a Tolosana,142,middle,False,71999127,ES
321,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140,middle,False,123044942,ES
464,10.1007/s10120-024-01504-7,Leticia Moreira,35,last,True,71999127,ES
588,10.1016/j.cgh.2020.11.002,Leticia Moreira,50,middle,False,71999127,ES
755,10.1016/j.jmoldx.2022.02.003,Mar Costa,84,middle,False,123044942,ES
...,...,...,...,...,...,...,...
184091,10.1016/j.jaip.2024.03.050,Sami Aqel,29,middle,False,123044942,ES
184486,10.1136/ard-2023-224990,Núria Guañabens,29,middle,False,71999127,ES
184636,10.1111/apt.18230,Xavier Calvet,29,middle,False,123044942,ES
186635,10.2139/ssrn.4139263,Albert Palou Vilar,31,middle,False,115304662,ES


In [31]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1007/s13304-024-01859-7,M Serrano-Navidad,None,<NA>,None
1,10.1007/s00192-022-05365-5,Carolina Vidal Gamboa,Public Health Institute of the Andrés Bello Un...,13897259,CL
2,10.1111/jcmm.16736,Lejla Pojskić,Institute for Genetic Engineering and Biotechn...,104817121,BA
3,10.1038/s41420-024-01857-z,Marina Mola-Caminal,"Neurology, Hospital del Mar Medical Research I...",4210130874,ES
4,10.1371/journal.pone.0265079,José Manuel Santos‐Lozano,Centro de Investigación Biomédica en Red Fisio...,4210166912,ES
...,...,...,...,...,...
32437,10.1016/j.gaceta.2021.01.005,Jorge Luis Díaz,Unidad Docente de Medicina Preventiva y Salud ...,170486558,ES
32438,10.1016/j.biopha.2024.116857,Anne Hansen Ree,"Oncobell Program, Bellvitge Institute for Biom...",2800192669,NO
32439,10.1016/j.biopha.2024.116857,Anne Hansen Ree,"ProCURE Program, Catalan Institute of Oncology...",2800192669,NO
32440,10.1016/j.biopha.2024.116857,Anne Hansen Ree,"University of Oslo, Problemveien 11, Oslo 0313...",2800192669,NO


In [32]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
# to_check.to_csv('to_check.csv')
to_check
### WE MISS 876 DOIS BY FILTERING CHECKING THE THRESHOLD OF 1 AND MANUALLY CHECK THE MISSING

1542


,DOI
raw_affiliation,
"Universitat Pompeu Fabra (UPF), Barcelona, Spain",68
"IMIM (Hospital del Mar Medical Research Institute), Barcelona, Spain",57
"ISGlobal, Barcelona, Spain",49
"Universitat Pompeu Fabra, Barcelona, Spain",41
"Department of Preventive Medicine, University of Valencia, Valencia, Spain",40
...,...
"Universidad Francisco de Vitoria, Madrid, Spain",2
"Bellvitge University Hospital, Department of Psychiatry, Bellvitge Biomedical Research Institute (IDIBELL), Neurosciences Group - Psychiatry and Mental Health, Barcelona, Spain",2
"Universitat Rovira i Virgili, Departament de Bioquímica i Biotecnologia, Alimentació, Nutrició, Desenvolupament i Salut Mental (ANUT-DSM), Unitat de Nutrició Humana, Reus, Spain",2


In [37]:
df_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '>1', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


0.5674351585014409

In [38]:
cerca_string = {'BETA' : ['vic', 'beta'], # NOT IN OA
                    'CREAF' : ['creaf', 'cerdanyola', 'bellaterra'],
                    'ICN2' : ['icn2', 'bellaterra', 'cerdanyola'],
                    'ISGlobal' : ['global'],
                    'ResearchMar' : ['mar', 'imim']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_tmp = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check.raw_affiliation))].reset_index()
df_tmp['raw_affiliation'] = df_tmp['raw_affiliation'].str.lower()

df_tmp_1 = df_tmp[df_tmp['raw_affiliation'].str.contains(('|'.join(cerca_string)), case=False, na=False)].drop_duplicates().reset_index(drop = True)
# df_tmp_1.to_csv('to_check_v2.csv')
df_tmp_1

,raw_affiliation,DOI
0,"internal medicine department, imim (hospital d...",1
1,integrative pharmacology and systems neuroscie...,1
2,integrative pharmacology and systems neuroscie...,1
3,integrated pharmacology and systems neuroscien...,1
4,integrative pharmacology and systems neuroscie...,1
...,...,...
4541,"department of neuroradiology, hospital del mar...",1
4542,"department of neuroradiology, hospital del mar...",1
4543,"department of neuropsychiatry and addictions, ...",1
4544,"department of neuropsychiatry and addictions, ...",1


In [39]:
df_check_v2 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '=1', col_names = True, row_names = False)
df_check_v2

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,raw_affiliation,DOI,CERCA,CERCA_IMIM
0,". clínica de rehabilitación de salud mental, d...",1,FALSE,
1,. department of child and adolescent psychiatr...,1,FALSE,FALSE
2,". department of psychiatry, chinese university...",1,FALSE,FALSE
3,. hospital del mar medical research institute ...,1,TRUE,
4,". infectious diseases service, hospital del ma...",1,TRUE,
...,...,...,...,...
4543,wildlife ecology & health group (we&h) and ser...,1,FALSE,
4544,"wildlife ecology & health group (we&h), and se...",1,FALSE,
4545,"wildlife ecology & health group (we&h), servei...",1,FALSE,
4546,wildlife ecology & health research group (we&h...,1,FALSE,


In [40]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_1_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]
check_2_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA_IMIM == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) +
    list(check_1_imim.DOI.unique()) +
    list(check_2_imim.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.6812680115273775

In [41]:
cerca_possible_authors_not_parent = df_ResearchMar[~df_ResearchMar.DOI.isin(cerca_doi)]

in_query = str(tuple(cerca_possible_authors_not_parent.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca_not_parent = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca_not_parent

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1177/19476035211053827,Demirhan Dıraçoğlu,Department of Physical Medicine and Rehabilita...,67581229,TR
1,10.1016/j.thromres.2021.06.008,Takao Kato,"Department of Cardiovascular Medicine, Graduat...",22299242,JP
2,10.1002/hem3.70004,Alicia Enrico,Hospital Italiano La Plata Buenos Aires Argentina,4210118619,AR
3,10.1002/hem3.70004,Deepesh Lad,Department of Internal Medicine Postgraduate I...,45294948,IN
4,10.1111/jdv.18574,Ece Nur Değirmentepe,"Okmeydani Training and Research Hospital, Ista...",4210155168,TR
...,...,...,...,...,...
17614,10.1093/hmg/ddac084,Stephen R. Braddock,"Division of Medical Genetics, Department of Pe...",47838141,US
17615,10.1002/alz.12662,Nancy S. Foldi,"Department of Psychiatry, New York University ...",40889946,US
17616,10.1002/alz.12662,Nancy S. Foldi,"Department of Psychology, Queens College and T...",40889946,US
17617,10.1016/j.esmoop.2024.103972,Jean Hoffman‐Censits,Johns Hopkins Sidney Kimmel Comprehensive Canc...,4210164401,US


In [42]:
grouped = df_possible_cerca_not_parent[df_possible_cerca_not_parent.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 0]
to_check.to_csv( 'to_check_v3.csv' )
to_check

,DOI
raw_affiliation,
"Hospital del Mar, Barcelona, Spain",41
"Hospital Universitario 12 de Octubre, Madrid, Spain",13
"Department of Dermatology, Hospital del Mar, Barcelona, Spain",13
"Hospital Universitario Central de Asturias, Oviedo, Spain",12
"Medical Oncology Department, Hospital del Mar, Barcelona, Spain",11
...,...
"Department of Surgery, Hospital Universitario Miguel Servet, 50009 Zaragoza, Spain",1
"Department of Surgery, Hospital Universitario Marqués de Valdecilla, 39008 Santander, Spain",1
"Department of Surgery, Hospital Universitario Lozano Blesa, 50009 Zaragoza, Spain",1


In [43]:
df_check_v3 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'Nonparent', col_names = True, row_names = False)
df_check_v3

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,raw_affiliation,DOI,CERCA
0,"Hospital del Mar, Barcelona, Spain",41,TRUE
1,"Department of Dermatology, Hospital del Mar, B...",13,TRUE
2,"Hospital Universitario 12 de Octubre, Madrid, ...",13,FALSE
3,"Hospital Universitario Central de Asturias, Ov...",12,FALSE
4,"Medical Oncology Department, Hospital del Mar,...",11,TRUE
...,...,...,...
8398,"Department of Surgery, Hospital Universitario ...",1,FALSE
8399,"Department of Surgery, Hospital Universitario ...",1,FALSE
8400,"Department of Surgery, Hospital Universitario ...",1,FALSE
8401,"Department of Surgery, Hospital Universitario ...",1,FALSE


In [44]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_1_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]
check_2_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_3 = df_possible_cerca_not_parent[df_possible_cerca_not_parent.raw_affiliation.isin(df_check_v3[df_check_v3.CERCA == 'TRUE'].raw_affiliation)]


cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) +
    list(check_1_imim.DOI.unique()) +
    list(check_2_imim.DOI.unique()) + 
    list(check_3.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.8435158501440922

In [45]:
df_check = pd.concat((cerca_authors, check_1, check_2, check_1_imim, check_2_imim, check_3))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
282,10.1088/2515-7639/acc893,Masoud Karimipour,35,middle,False,4210093216,ES,NaN
593,10.1016/j.jval.2021.03.021,Luis Gları́a,47,middle,False,4210156109,ES,NaN
739,10.1016/j.euroneuro.2024.05.006,Alba Toll,35,middle,False,4210156109,ES,NaN
915,10.1007/s10654-021-00733-9,Martine Vrijheid,35,middle,False,4210148332,ES,NaN
934,10.1111/acel.14194,Martine Vrijheid,52,middle,False,4210148332,ES,NaN
...,...,...,...,...,...,...,...,...
14231,10.1093/bjd/ljae067,Ana M. Giménez‐Arnau,<NA>,NaN,<NA>,4210130874,ES,"Hospital Del Mar Research Institute, Universit..."
14238,10.1016/j.transproceed.2021.09.013,Marta Crespo,<NA>,NaN,<NA>,4210130874,ES,"Department of Nephrology, Hospital del Mar, Ba..."
14241,10.1016/j.avsg.2021.10.054,Albert Benet i Clarà,<NA>,NaN,<NA>,4210115082,ES,Department of Vascular and Endovascular Surger...
14246,10.1016/j.chest.2023.04.029,Esther Barreiro,<NA>,NaN,<NA>,2801357902,ES,"Servicio de Neumología, Hospital del Mar-IMIM,..."


In [46]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_final = df_OA.merge(df_centers, on = 'DOI').drop_duplicates().reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv', index = False)
df_final

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,4210154894,ES,False,ResearchMar
1,10.1038/s41591-023-02610-2,Charles Agyemang,47,middle,False,887064364,NL,False,ResearchMar
2,10.1038/s41591-023-02610-2,Andrew Wong,750,middle,False,45129253,GB,False,ResearchMar
3,10.1038/s41591-023-02610-2,Farhad Zamani,766,middle,False,161106909,IR,False,ResearchMar
4,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,63634437,ES,False,ResearchMar
...,...,...,...,...,...,...,...,...,...
193398,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210133907,FR,False,ISGlobal
193399,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210097159,FR,False,ISGlobal
193400,10.1111/jvh.13412,Karine Lacombe,33,middle,False,39804081,FR,False,ISGlobal
193401,10.1002/lol2.10277,Maria Lundgren,33,middle,False,223464139,SE,False,BETA


In [47]:
df_final[df_final.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             50
CREAF           825
ICN2            712
ISGlobal        626
ResearchMar    3725
dtype: int64